# Demographic conditioning (ANES 2020, GPT-4.1-mini, Replicate)


In [15]:
import pandas as pd
from collections import Counter
import numpy as np
from scipy.stats import entropy

In [16]:
def jsd_divergence_between_sets(set1, set2, base=2):

    jsd_results = {}

    dict1 = {list(d.keys())[0]: list(d.values())[0] for d in set1}
    dict2 = {list(d.keys())[0]: list(d.values())[0] for d in set2}

    for question in dict1.keys():
        if question not in dict2:
            continue 

        p_counts = dict1[question]
        q_counts = dict2[question]

        p = np.array([p_counts.get(k, 0) for k in p_counts], dtype=float)
        q = np.array([q_counts.get(k, 0) for k in p_counts], dtype=float)

        p /= p.sum()
        q /= q.sum()

        epsilon = 1e-10
        p = np.clip(p, epsilon, 1)
        q = np.clip(q, epsilon, 1)

        m = 0.5 * (p + q)
        jsd_pq = 0.5 * (entropy(p, m, base=base) + entropy(q, m, base=base)) 
        jsd_results[question] = jsd_pq


    return jsd_results

In [17]:
group_var_dict = {"V201549x" : "Race", 'V202022' : "Discuss Politics", 'V201200' : "Ideology", 'V201231x' : "Party", 
                  'V201452' : "Church", 'V201600' : "Gender", 'V202406' : "Political Interest"}

groups = {
    # 'V201549x': {1: 'White', 2: 'Black', 3: 'Hispanic', 4: 'Asian', 5: 'Native American'},
    'V201549x': {1: 'White', 2: 'Black', 3: 'Asian', 4: 'Native American', 5: 'Hispanic'},
    'V202022': {
            1: 'Like to discuss politics',
            2: 'Never discuss politics'
            },
    'V201200': {
            1: "Extremely liberal",
            2: "Liberal",
            3: "Slightly liberal",
            4: "Moderate",
            5: "Slightly conservative",
            6: "Conservative",
            7: "Extremely conservative"
        },
    'V201231x': {
            1: "Strong democrat",
            2: "Weak democrat",
            3: "Democrat-leaning indep.",
            4: "Independent",
            5: "Republican-leaning indep.",
            6: "Weak republican",
            7: "Strong republican"
        },
    'V201452': {1: "Attend church", 2: "Does not attend church"},
    'V201600': {1: "Man", 2: "Woman"},
    'V202406': {1: "Very", 2: "Somewhat", 3: "Not very", 4: "Not at all"}
}

label_to_group = {}

for var, mapping in groups.items():
    for code, label in mapping.items():
        label_to_group[label] = group_var_dict[var]

In [18]:
qids = ["V201324","V201416", "V202234", "V202257", "V202287", "V202332", "V202337", "V202348", "V202371", "V202378"]
name_dict = {"V201324" : "Current Economy", "V201416" : "Gay Marriage", "V202234" : "Refugee Allowing", 
            "V202257" : "Income Inequality", "V202287" : "Gender Role", "V202332": "Climate Change",
            "V202337": "Gun Regulation", "V202348": "Drug Addiction", "V202371": "Race Diversity", "V202378": "Health Insurance"}
qids_reverse_coded = ["V201416", "V202234", "V202257", "V202287", "V202332", "V202371"]

groups_keys = groups.keys()

def get_counts_group(path, qids, key): #function to get a dict with counts for each question

    value_list = {}

    for number, group in groups[key].items():

        value_list[group] = []

        for id in qids:
            df = pd.read_csv(path + id + ".csv")
            df_group = df[df[key] == number]

            df_group.loc[:, 'Response'] = pd.to_numeric(df_group['Response'], errors='coerce')     
            value_counts = Counter(df_group['Response'].dropna())

            if id in ["V201324", "V202332"]:
                answers = {i: value_counts[i] for i in range(1, 6)}
            else:
                answers = {i: value_counts[i] for i in range(1, 4)}

            value_list[group].append({name_dict[id] : answers})

    return value_list

In [19]:
def get_human_counts_group(path, qids, key): #getting results from anes 2020
    value_list = {}

    for number, group in groups[key].items():
        value_list[group] = []
        df = pd.read_csv(path)
        df_group = df[df[key] == number]

        for id in qids:
            human_counts = Counter(df_group[id].dropna())

            if id in ["V201324", "V202332"]:
                answers = {i: human_counts[i] for i in range(1, 6)}
            else:
                answers = {i: human_counts[i] for i in range(1, 4)}

            value_list[group].append({name_dict[id] : answers})

    return value_list

#human_results = get_human_counts_group("data/2020 ANES_test.csv", qids)

## Load human (ANES 2020) and Replicate

Conditioning is checked under Replicate only, where the profile is presented as written.

In [20]:
model_name = "gpt4.1"
_MINI = "../Publication Results/Gpt4.1-Mini"

human_groups     = {}
replicate_groups = {}
for key in groups_keys:
    human_groups[group_var_dict[key]]     = get_human_counts_group("../data/2020 ANES_test.csv", qids, key)
    replicate_groups[group_var_dict[key]] = get_counts_group(f"{_MINI}/main_mq_results/full_results_2020_", qids, key)

ITEMS_ALL = list(name_dict.values())
print("variables:", list(human_groups.keys()))
print("items:", len(ITEMS_ALL))

/var/folders/8s/b_t8p6s52r9c8k1j22zs0l740000gn/T/ipykernel_50507/624287208.py:6: DtypeWarning: Columns (19,21,22,23,25,26,27,29,30,35,37,38,1508,1509) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)
/var/folders/8s/b_t8p6s52r9c8k1j22zs0l740000gn/T/ipykernel_50507/624287208.py:6: DtypeWarning: Columns (19,21,22,23,25,26,27,29,30,35,37,38,1508,1509) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)
/var/folders/8s/b_t8p6s52r9c8k1j22zs0l740000gn/T/ipykernel_50507/624287208.py:6: DtypeWarning: Columns (19,21,22,23,25,26,27,29,30,35,37,38,1508,1509) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)
/var/folders/8s/b_t8p6s52r9c8k1j22zs0l740000gn/T/ipykernel_50507/624287208.py:6: DtypeWarning: Columns (19,21,22,23,25,26,27,29,30,35,37,38,1508,1509) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


variables: ['Race', 'Discuss Politics', 'Ideology', 'Party', 'Church', 'Gender', 'Political Interest']
items: 10


## Add Age

Binned into decades. Any binning is sufficient to detect the presence or absence of conditioning: $G_k$ is identically zero when the silicon distribution does not vary with the variable, whatever bins are used.

In [21]:
AGE_COL, AGE_BINS = "V201507x", [17, 29, 39, 49, 59, 69, 120]
AGE_LABELS = {1:"18-29", 2:"30-39", 3:"40-49", 4:"50-59", 5:"60-69", 6:"70+"}

def _age_bin(df):
    v = pd.to_numeric(df[AGE_COL], errors="coerce").where(lambda s: s.between(18, 119))
    return pd.cut(v, bins=AGE_BINS, labels=list(AGE_LABELS)).astype(float)

def _counts_age(loader_path, qids, human=False, labels=AGE_LABELS):
    out = {g: [] for g in labels.values()}
    for qid in qids:
        df = pd.read_csv(loader_path if human else f"{loader_path}{qid}.csv")
        df["_bin"] = _age_bin(df)
        col = qid if human else "Response"
        df[col] = pd.to_numeric(df[col], errors="coerce")
        codes = range(1, 6) if qid in ["V201324", "V202332"] else range(1, 4)
        for num, g in labels.items():
            c = Counter(df.loc[df["_bin"] == num, col].dropna())
            out[g].append({name_dict[qid]: {i: c[i] for i in codes}})
    return out

human_groups["Age"]     = _counts_age("../data/2020 ANES_test.csv", qids, human=True)
replicate_groups["Age"] = _counts_age(f"{_MINI}/main_mq_results/full_results_2020_", qids)

print("variables:", list(human_groups.keys()))

/var/folders/8s/b_t8p6s52r9c8k1j22zs0l740000gn/T/ipykernel_50507/636769476.py:11: DtypeWarning: Columns (19,21,22,23,25,26,27,29,30,35,37,38,1508,1509) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(loader_path if human else f"{loader_path}{qid}.csv")
/var/folders/8s/b_t8p6s52r9c8k1j22zs0l740000gn/T/ipykernel_50507/636769476.py:11: DtypeWarning: Columns (19,21,22,23,25,26,27,29,30,35,37,38,1508,1509) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(loader_path if human else f"{loader_path}{qid}.csv")
/var/folders/8s/b_t8p6s52r9c8k1j22zs0l740000gn/T/ipykernel_50507/636769476.py:11: DtypeWarning: Columns (19,21,22,23,25,26,27,29,30,35,37,38,1508,1509) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(loader_path if human else f"{loader_path}{qid}.csv")
/var/folders/8s/b_t8p6s52r9c8k1j22zs0l740000gn/T/ipykernel_50507/636769476.py:11: DtypeWarning: Columns (1

variables: ['Race', 'Discuss Politics', 'Ideology', 'Party', 'Church', 'Gender', 'Political Interest', 'Age']


## $G_k$ and $B_k$

In [22]:
import numpy as np

def _to_dict(count_list):
    return {list(d.keys())[0]: list(d.values())[0] for d in count_list}


def _jsd_vec(P, Q, base=2, eps=1e-10):
    """Vectorised JSD over the last axis. Reproduces jsd_divergence_between_sets to ~1e-16."""
    P = np.asarray(P, float); Q = np.asarray(Q, float)
    P = P / P.sum(-1, keepdims=True); Q = Q / Q.sum(-1, keepdims=True)
    P = np.clip(P, eps, 1.0);         Q = np.clip(Q, eps, 1.0)
    P = P / P.sum(-1, keepdims=True); Q = Q / Q.sum(-1, keepdims=True)
    M = 0.5 * (P + Q); M = M / M.sum(-1, keepdims=True)
    return 0.5 * (np.sum(P * np.log(P / M), -1) + np.sum(Q * np.log(Q / M), -1)) / np.log(base)


def _cells(nested, variable, item):
    """{level: count vector} for one variable and one item."""
    out = {}
    for cat, lst in nested[variable].items():
        d = _to_dict(lst)
        if item in d:
            out[cat] = np.array([float(d[item][c]) for c in sorted(d[item])])
    return out


def _levels(human, silicon, variable, items, min_n):
    """Levels present in both samples and clearing min_n on every item, so the level set is
    identical across items and G_k averages over a fixed set."""
    keep, dropped = [], {}
    for c in [c for c in human[variable] if c in silicon[variable]]:
        n_h = min(_cells(human, variable, it)[c].sum() for it in items)
        n_s = min(_cells(silicon, variable, it)[c].sum() for it in items)
        (keep.append(c) if (n_h >= min_n and n_s >= min_n)
         else dropped.setdefault(c, (int(n_s), int(n_h))))
    return keep, dropped


def bh_adjust(p):
    p = np.asarray(p, float); n = len(p)
    order = np.argsort(p); adj = np.empty(n); prev = 1.0
    for rank, idx in zip(range(n, 0, -1), order[::-1]):
        prev = min(prev, p[idx] * n / rank)
        adj[idx] = prev
    return adj


def _boot(A, n_boot, rng):
    """counts (L, C) -> (n_boot, L, C) multinomial resamples, cell sizes fixed."""
    out = np.empty((n_boot,) + A.shape)
    for i, v in enumerate(A):
        n = int(round(v.sum()))
        p = v / v.sum() if v.sum() > 0 else np.full(len(v), 1.0 / len(v))
        out[:, i, :] = rng.multinomial(n, p, size=n_boot)
    return out


def conditioning(human, silicon, items=None, min_n=30, n_boot=2000, seed=0, alpha=0.05):
    """Per-variable G_k and B_k, averaged over items, with a percentile bootstrap on G_k."""
    items = ITEMS_ALL if items is None else items
    rng = np.random.default_rng(seed)
    summary, per_item = [], []
    for variable in human:
        cats, dropped = _levels(human, silicon, variable, items, min_n)
        if dropped:
            print(f"[{variable}] dropped at min_n={min_n}: "
                  + ", ".join(f"{c} (silicon {a}, human {b})" for c, (a, b) in dropped.items()))
        if len(cats) < 2:
            print(f"[{variable}] skipped: fewer than two usable levels"); continue
        L = len(cats); eye = np.eye(L, dtype=bool); iu = np.triu_indices(L, 1)
        g_item, b_item = [], []
        boot_g = np.zeros((n_boot, len(items)))
        for j, item in enumerate(items):
            H = _cells(human, variable, item); S = _cells(silicon, variable, item)
            Hm = np.stack([H[c] for c in cats]); Sm = np.stack([S[c] for c in cats])
            M = _jsd_vec(Sm[:, None, :], Hm[None, :, :])
            off = M.copy(); np.fill_diagonal(off, np.nan)
            g_item.append((np.nanmean(off, 1) - np.diag(M)).mean())
            b_item.append(_jsd_vec(Hm[:, None, :], Hm[None, :, :])[iu].mean())
            per_item.append(dict(variable=variable, item=item,
                                 G=g_item[-1], B=b_item[-1]))
            Hb = _boot(Hm, n_boot, rng); Sb = _boot(Sm, n_boot, rng)
            Mb = _jsd_vec(Sb[:, :, None, :], Hb[:, None, :, :])
            boot_g[:, j] = (np.nanmean(np.where(eye[None], np.nan, Mb), 2) - Mb[:, eye]).mean(1)
        bg = boot_g.mean(1)
        lo, hi = np.percentile(bg, [100*alpha/2, 100*(1-alpha/2)])
        p = min(max(2 * min((bg <= 0).mean(), (bg >= 0).mean()), 1.0/n_boot), 1.0)
        summary.append(dict(variable=variable, levels=L, B=np.mean(b_item),
                            G=np.mean(g_item), G_lo=lo, G_hi=hi, p_raw=p))
    S = pd.DataFrame(summary)
    if len(S):
        S["p_bh"] = bh_adjust(S["p_raw"].values)
        S["sig"] = S["p_bh"] < alpha
        S = S.sort_values("G", ascending=False).reset_index(drop=True)
    return S, pd.DataFrame(per_item)

In [23]:
MIN_N, N_BOOT = 30, 2000

cond, cond_items = conditioning(human_groups, replicate_groups, min_n=MIN_N, n_boot=N_BOOT, seed=0)
display(cond[["variable", "levels", "B", "G", "G_lo", "G_hi", "p_bh", "sig"]].round(4))

,variable,levels,B,G,G_lo,G_hi,p_bh,sig
0,Ideology,7,0.1321,0.0851,0.0822,0.0882,0.0008,True
1,Party,7,0.0778,0.0616,0.0598,0.0635,0.0008,True
2,Political Interest,4,0.0133,0.0080,0.0067,0.0092,0.0008,True
3,Church,2,0.0178,0.0033,0.0024,0.0042,0.0008,True
4,Gender,2,0.0032,0.0011,0.0007,0.0017,0.0008,True
5,Age,6,0.0070,0.0007,0.0003,0.0012,0.0013,True
6,Race,5,0.0129,-0.0002,-0.0017,0.0013,0.8550,False
7,Discuss Politics,2,0.0164,-0.0002,-0.0019,0.0015,0.8550,False


## Ablation

Three additional Replicate runs. The ablation prompts omit the dropped attributes, so those runs
cannot be grouped by them; each is loaded only by the variables it retains.

In [ ]:
_VAR_TO_KEY = {group_var_dict[k]: k for k in groups_keys}
_ALL_VARS = list(_VAR_TO_KEY)
_RUNS = {
    "no_demo":           ("main_mq_results_no_demo",           []),
    "no_ideology":       ("main_mq_results_no_ideology",       [v for v in _ALL_VARS if v != "Ideology"]),
    "no_ideology_party": ("main_mq_results_no_ideology_party", [v for v in _ALL_VARS if v not in ("Ideology", "Party")]),
}

def pooled_counts(prefix, qids):
    """One {item: {code: count}} over all respondents, no demographic filtering."""
    out = []
    for qid in qids:
        df = pd.read_csv(prefix + qid + ".csv")
        df.loc[:, "Response"] = pd.to_numeric(df["Response"], errors="coerce")
        vc = Counter(df["Response"].dropna())
        codes = range(1, 6) if qid in ["V201324", "V202332"] else range(1, 4)
        out.append({name_dict[qid]: {i: vc[i] for i in codes}})
    return out

def pooled_from_nested(nested):
    var = next(iter(nested)); acc = {}
    for cat, lst in nested[var].items():
        for entry in lst:
            (item, counts), = entry.items()
            d = acc.setdefault(item, {})
            for c, n in counts.items():
                d[c] = d.get(c, 0) + n
    return [{item: counts} for item, counts in acc.items()]

pooled = {"full_profile": pooled_from_nested(replicate_groups),
          "human":        pooled_from_nested(human_groups)}
ablation_groups = {}
for name, (subdir, retained) in _RUNS.items():
    prefix = f"{_MINI}/{subdir}/full_results_2020_"
    pooled[name] = pooled_counts(prefix, qids)
    ablation_groups[name] = {v: get_counts_group(prefix, qids, _VAR_TO_KEY[v]) for v in retained}
    if retained:                      # age is not dropped by any ablation prompt
        try:
            ablation_groups[name]["Age"] = _counts_age(prefix, qids)
        except KeyError:
            print(f"  ({name}: no {AGE_COL} column, age skipped)")
    print(f"{name}: grouped by {list(ablation_groups[name]) or 'nothing (pooled only)'}")

for name, pl in pooled.items():
    print(f"{name:20s} total responses = {sum(sum(v.values()) for d in pl for v in d.values())}")

no_demo: grouped by nothing (pooled only)
no_ideology: grouped by ['Race', 'Discuss Politics', 'Party', 'Church', 'Gender', 'Political Interest', 'Age']
no_ideology_party: grouped by ['Race', 'Discuss Politics', 'Church', 'Gender', 'Political Interest', 'Age']
full_profile         total responses = 51820
human                total responses = 46418
no_demo              total responses = 54409
no_ideology          total responses = 54410
no_ideology_party    total responses = 54406


Totals should be comparable across runs. If they differ substantially, the runs drew different profile samples and the aggregate comparison below carries sampling noise.

In [26]:
_ORDER = ["full_profile", "no_ideology", "no_ideology_party", "no_demo"]
_NICE = {"full_profile": "Full profile", "no_ideology": "Ideology removed",
         "no_ideology_party": "Ideology and party removed", "no_demo": "No demographics"}

rows = {}
for name in _ORDER:
    j = jsd_divergence_between_sets(pooled[name], pooled["human"])
    rows[_NICE[name]] = {**j, "MEAN": np.mean(list(j.values()))}
ablation = pd.DataFrame(rows).T

print("Aggregate JS-divergence from the human distribution")
display(ablation[["MEAN"]].round(4))
print("\nPer item")
display(ablation.drop(columns="MEAN").round(3))

_b, _n = ablation.loc["Full profile", "MEAN"], ablation.loc["No demographics", "MEAN"]
print(f"\nfull profile {_b:.3f} -> no demographics {_n:.3f}   (difference {_n - _b:+.3f})")

Aggregate JS-divergence from the human distribution


,MEAN
Full profile,0.0904
Ideology removed,0.0981
Ideology and party removed,0.1857
No demographics,0.2652



Per item


,Current Economy,Gay Marriage,Refugee Allowing,Income Inequality,Gender Role,Climate Change,Gun Regulation,Drug Addiction,Race Diversity,Health Insurance
Full profile,0.197,0.038,0.081,0.038,0.071,0.053,0.002,0.162,0.227,0.035
Ideology removed,0.202,0.054,0.115,0.056,0.060,0.048,0.003,0.164,0.230,0.050
Ideology and party removed,0.238,0.128,0.222,0.254,0.203,0.142,0.061,0.171,0.259,0.180
No demographics,0.249,0.173,0.277,0.341,0.203,0.417,0.284,0.172,0.269,0.268



full profile 0.090 -> no demographics 0.265   (difference +0.175)


### Does Party still condition once Ideology is removed?

In [27]:
def G_one(human, silicon, variable, items=None, min_n=MIN_N):
    """Point estimate of G_k for a single variable."""
    items = ITEMS_ALL if items is None else items
    cats, _ = _levels(human, silicon, variable, items, min_n)
    if len(cats) < 2:
        return np.nan
    vals = []
    for item in items:
        H = _cells(human, variable, item); S = _cells(silicon, variable, item)
        M = _jsd_vec(np.stack([S[c] for c in cats])[:, None, :],
                     np.stack([H[c] for c in cats])[None, :, :])
        off = M.copy(); np.fill_diagonal(off, np.nan)
        vals.append((np.nanmean(off, 1) - np.diag(M)).mean())
    return float(np.mean(vals))

g_full   = G_one(human_groups, replicate_groups, "Party")
g_noideo = G_one(human_groups, ablation_groups["no_ideology"], "Party")
print(f"G_party, full profile     = {g_full:.4f}")
print(f"G_party, ideology removed = {g_noideo:.4f}")

# nothing should spuriously appear for the null variables once the political ones are dropped
chk = []
for v in ["Political Interest", "Church", "Gender", "Age", "Race", "Discuss Politics"]:
    row = {"variable": v, "full": round(G_one(human_groups, replicate_groups, v), 4)}
    for name in ["no_ideology", "no_ideology_party"]:
        sil = ablation_groups[name]
        row[name] = round(G_one(human_groups, sil, v), 4) if v in sil else np.nan
    chk.append(row)
display(pd.DataFrame(chk))

G_party, full profile     = 0.0616
G_party, ideology removed = 0.0739


,variable,full,no_ideology,no_ideology_party
0,Political Interest,0.0080,0.0097,0.0185
1,Church,0.0033,0.0034,0.0034
2,Gender,0.0011,0.0007,0.0027
3,Age,0.0007,0.0013,0.0003
4,Race,-0.0002,0.0013,0.0011
5,Discuss Politics,-0.0002,0.0031,0.0047


### Save

In [ ]:
import os
outdir = "../tables-stochastic/conditioning"
os.makedirs(outdir, exist_ok=True)

cond.round(4).to_csv(f"{outdir}/{model_name}_conditioning.csv", index=False)
cond_items.round(4).to_csv(f"{outdir}/{model_name}_conditioning_per_item.csv", index=False)
ablation.round(4).to_csv(f"{outdir}/{model_name}_ablation.csv")
print("saved to", outdir)

In [29]:
# how much does the aggregate move under profile resampling alone?
def _pooled_from_nested(nested):
    """Sum over the levels of one variable to get the pooled {item: {code: count}}."""
    var = next(iter(nested))
    acc = {}
    for cat, lst in nested[var].items():
        for entry in lst:
            (item, counts), = entry.items()
            d = acc.setdefault(item, {})
            for c, n in counts.items():
                d[c] = d.get(c, 0) + n
    return [{item: counts} for item, counts in acc.items()]

rng = np.random.default_rng(0)
_p = _pooled_from_nested(replicate_groups)          # full-profile pooled counts
_h = _pooled_from_nested(human_groups)
_base = np.mean(list(jsd_divergence_between_sets(_p, _h).values()))
_boot = []
for _ in range(500):
    _rs = [{it: dict(zip(sorted(c), rng.multinomial(sum(c.values()),
            np.array([c[k] for k in sorted(c)], float) / sum(c.values()))))}
           for d in _p for it, c in d.items()]
    _boot.append(np.mean(list(jsd_divergence_between_sets(_rs, _h).values())))
print(f"aggregate JSD {_base:.4f}, resampling sd {np.std(_boot):.4f}, "
      f"95% range {np.percentile(_boot,2.5):.4f}-{np.percentile(_boot,97.5):.4f}")

aggregate JSD 0.0904, resampling sd 0.0008, 95% range 0.0890-0.0921
